# Fairness Audit of Monocular 3D Face Mesh Estimation
**Authors:** Pranay Sharma, Srishty Jain

**Question:** Does a widely used single-image 3D face model (MediaPipe Face Landmarker, 478 3D landmarks) work equally reliably for people of different race, sex and age?

**What we measure (no 3D ground truth is needed):**
1. **Failure rate**: how often no face mesh is produced, per group.
2. **Robustness (NME)**: how much the 3D mesh moves when the same photo is made darker, brighter, low-contrast, blurred, JPEG-compressed or rotated by ±10°, normalised by inter-ocular distance.
3. **FASD-relevant measurements**: stability of palpebral fissure length, philtrum length and upper-lip thickness under the same changes.

**Data:** FairFace validation split (balanced across 7 race groups, including *Indian*).

**How to run:** Runtime → Run all. Takes ~10–20 min on CPU. At the end, download `results.zip` and send it back for the paper.

In [ ]:
!pip -q install mediapipe datasets opencv-python-headless scipy pandas matplotlib
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!ls -la face_landmarker.task

## 1. Settings

In [ ]:
import json, numpy as np, pandas as pd, cv2
N_PER_RACE = 300     # images per race group (7 groups). Use 100 for a quick test run.
UPSCALE = 2          # FairFace images are 224px; upscale for more reliable landmarks
SEED = 42
OUT = "results"
import os; os.makedirs(OUT, exist_ok=True)

## 2. Core functions (perturbations and metrics)

In [ ]:
import numpy as np, cv2

# MediaPipe Face Mesh landmark indices
EYE_R_OUT, EYE_R_IN, EYE_L_IN, EYE_L_OUT = 33, 133, 362, 263
SUBNASALE, UPPER_LIP_TOP, UPPER_LIP_BOTTOM = 2, 0, 13

# ---------- perturbations: each returns (image, inverse_fn for landmark xy in pixels) ----------
def _id(xy, w, h): return xy
def p_dark(img):   return cv2.convertScaleAbs(img, alpha=0.5, beta=0), _id
def p_bright(img): return cv2.convertScaleAbs(img, alpha=1.5, beta=10), _id
def p_lowcontrast(img):
    m = img.mean(); return np.clip((img - m) * 0.5 + m, 0, 255).astype(np.uint8), _id
def p_blur(img):   return cv2.GaussianBlur(img, (0, 0), 2.0), _id
def p_jpeg(img):
    ok, b = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 20]); return cv2.imdecode(b, cv2.IMREAD_COLOR), _id
def _rot(img, ang):
    h, w = img.shape[:2]; M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
    out = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    Minv = cv2.invertAffineTransform(M)
    def inv(xy, w_, h_):
        pts = np.hstack([xy[:, :2], np.ones((len(xy), 1))]) @ Minv.T
        out_xy = xy.copy(); out_xy[:, :2] = pts; return out_xy
    return out, inv
def p_rot_p10(img): return _rot(img, 10)
def p_rot_m10(img): return _rot(img, -10)

PERTURBATIONS = {"dark": p_dark, "bright": p_bright, "low_contrast": p_lowcontrast,
                 "blur": p_blur, "jpeg_q20": p_jpeg, "rot+10": p_rot_p10, "rot-10": p_rot_m10}

# ---------- metrics ----------
def to_pixels(lms, w, h):
    """lms: (N,3) normalized MediaPipe landmarks -> pixel units (z scaled by width, as in MediaPipe)."""
    a = np.asarray(lms, dtype=float).copy(); a[:, 0] *= w; a[:, 1] *= h; a[:, 2] *= w; return a

def interocular(p): return np.linalg.norm(p[EYE_R_OUT, :2] - p[EYE_L_OUT, :2])

def nme(p_ref, p_pert, use_z=True):
    """Mean landmark displacement normalised by inter-ocular distance."""
    k = 3 if use_z else 2
    return float(np.linalg.norm(p_ref[:, :k] - p_pert[:, :k], axis=1).mean() / interocular(p_ref))

def fasd_measures(p):
    """FASD-relevant facial measurements, normalised by inter-ocular distance."""
    io = interocular(p)
    pfl = (np.linalg.norm(p[EYE_R_OUT] - p[EYE_R_IN]) + np.linalg.norm(p[EYE_L_IN] - p[EYE_L_OUT])) / 2
    philtrum = np.linalg.norm(p[SUBNASALE] - p[UPPER_LIP_TOP])
    lip = np.linalg.norm(p[UPPER_LIP_TOP] - p[UPPER_LIP_BOTTOM])
    return {"pfl": pfl / io, "philtrum": philtrum / io, "upper_lip": lip / io}


## 3. Statistics helpers

In [ ]:
import numpy as np, pandas as pd
from scipy import stats

def boot_ci(x, n=1000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    m = rng.choice(x, (n, len(x)), replace=True).mean(1)
    return tuple(np.percentile(m, [2.5, 97.5]))

def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); adj = np.empty_like(p); run = 0
    for r, i in enumerate(order):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(run, 1.0)
    return adj

def group_report(df, group_col, value_col):
    """Per-group mean with bootstrap CI, Kruskal-Wallis across groups, and
    Mann-Whitney of every group vs the best (lowest-median) group, Holm-corrected."""
    d = df[[group_col, value_col]].dropna()
    rows = []
    for g, s in d.groupby(group_col)[value_col]:
        lo, hi = boot_ci(s.values)
        rows.append({group_col: g, "n": len(s), "mean": s.mean(), "median": s.median(), "ci_low": lo, "ci_high": hi})
    t = pd.DataFrame(rows).sort_values("median").reset_index(drop=True)
    samples = [s.values for _, s in d.groupby(group_col)[value_col]]
    kw = stats.kruskal(*samples) if len(samples) > 1 else None
    best = t.loc[0, group_col]; ref = d[d[group_col] == best][value_col].values
    ps = [np.nan if g == best else stats.mannwhitneyu(d[d[group_col] == g][value_col].values, ref, alternative="two-sided").pvalue
          for g in t[group_col]]
    mask = ~np.isnan(ps); adj = np.full(len(ps), np.nan); adj[mask] = holm(np.array(ps)[mask])
    t["p_vs_best_holm"] = adj
    t["ratio_vs_best"] = t["mean"] / t.loc[0, "mean"]
    return t, (kw.statistic, kw.pvalue) if kw else (np.nan, np.nan)

def failure_report(df, group_col, fail_col="failed"):
    ct = pd.crosstab(df[group_col], df[fail_col])
    rate = df.groupby(group_col)[fail_col].mean().rename("failure_rate")
    chi = stats.chi2_contingency(ct) if ct.shape[1] > 1 else None
    return rate, (chi[0], chi[1]) if chi else (np.nan, np.nan)


## 4. Load FairFace and draw a balanced sample

In [ ]:
from datasets import load_dataset
ds = load_dataset("HuggingFaceM4/FairFace", "0.25", split="validation")
race_names = ds.features["race"].names; gender_names = ds.features["gender"].names; age_names = ds.features["age"].names
print(len(ds), race_names)
rng = np.random.default_rng(SEED)
races = np.array(ds["race"])
idx = np.concatenate([rng.choice(np.where(races == r)[0], min(N_PER_RACE, (races == r).sum()), replace=False)
                      for r in range(len(race_names))])
sample = ds.select(sorted(idx.tolist()))
print("sampled", len(sample))

## 5. Run the 3D face model on original and perturbed images

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python as mpt
from mediapipe.tasks.python import vision
from tqdm.auto import tqdm
det = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mpt.BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=vision.RunningMode.IMAGE, num_faces=1))

def landmarks(rgb):
    r = det.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb)))
    if not r.face_landmarks: return None
    return np.array([[l.x, l.y, l.z] for l in r.face_landmarks[0]])

rows = []
for i, ex in enumerate(tqdm(sample)):
    rgb = np.array(ex["image"].convert("RGB"))
    rgb = cv2.resize(rgb, None, fx=UPSCALE, fy=UPSCALE, interpolation=cv2.INTER_CUBIC)
    h, w = rgb.shape[:2]
    base = {"id": i, "race": race_names[ex["race"]], "gender": gender_names[ex["gender"]], "age": age_names[ex["age"]]}
    l0 = landmarks(rgb)
    if l0 is None:
        rows.append({**base, "perturbation": "original", "failed": True}); continue
    p0 = to_pixels(l0, w, h); m0 = fasd_measures(p0)
    rows.append({**base, "perturbation": "original", "failed": False, **{f"{k}": v for k, v in m0.items()}})
    for name, fn in PERTURBATIONS.items():
        img2, inv = fn(rgb); l1 = landmarks(img2)
        if l1 is None:
            rows.append({**base, "perturbation": name, "failed": True}); continue
        p1 = inv(to_pixels(l1, w, h), w, h); m1 = fasd_measures(p1)
        rows.append({**base, "perturbation": name, "failed": False, "nme": nme(p0, p1), "nme_2d": nme(p0, p1, use_z=False),
                     **{f"d_{k}": abs(m1[k] - m0[k]) / m0[k] for k in m0}})
df = pd.DataFrame(rows); df.to_csv(f"{OUT}/raw_results.csv", index=False)
print(df.groupby("perturbation")["failed"].mean())

## 6. Analysis by race, gender and age

In [ ]:
age_bin = {"0-2":"0-19","3-9":"0-19","10-19":"0-19","20-29":"20-39","30-39":"20-39","40-49":"40-59","50-59":"40-59","60-69":"60+","more than 70":"60+"}
df["age_group"] = df["age"].map(age_bin).fillna(df["age"])
pert = df[df.perturbation != "original"].copy()
per_img = pert.groupby(["id", "race", "gender", "age_group"], as_index=False)[["nme", "nme_2d", "d_pfl", "d_philtrum", "d_upper_lip"]].mean()
summary = {}
for g in ["race", "gender", "age_group"]:
    rate, chi = failure_report(df[df.perturbation == "original"], g)
    rate.to_csv(f"{OUT}/failure_{g}.csv"); summary[f"failure_chi2_p_{g}"] = chi[1]
    for v in ["nme", "d_pfl", "d_philtrum", "d_upper_lip"]:
        t, kw = group_report(per_img, g, v); t.to_csv(f"{OUT}/{v}_by_{g}.csv", index=False)
        summary[f"{v}_by_{g}_kruskal_p"] = kw[1]
        summary[f"{v}_by_{g}_worst_best_ratio"] = float(t["ratio_vs_best"].max())
    print(f"\n=== {g} ==="); print(rate.round(3)); print(group_report(per_img, g, "nme")[0].round(4))
# race x perturbation table
rp = pert.groupby(["perturbation", "race"])["nme"].mean().unstack(); rp.to_csv(f"{OUT}/nme_race_x_perturbation.csv")
print(rp.round(4))
json.dump({k: (None if pd.isna(v) else float(v)) for k, v in summary.items()}, open(f"{OUT}/summary.json", "w"), indent=2)

## 7. Figures

In [ ]:
import matplotlib.pyplot as plt
t, _ = group_report(per_img, "race", "nme")
plt.figure(figsize=(7, 3.5))
plt.errorbar(t["race"], t["mean"], yerr=[t["mean"] - t["ci_low"], t["ci_high"] - t["mean"]], fmt="o", capsize=4)
plt.ylabel("Mean landmark shift (NME)"); plt.xticks(rotation=30, ha="right"); plt.title("Robustness of 3D face mesh by race group (95% CI)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_nme_by_race.png", dpi=200); plt.show()
plt.figure(figsize=(8, 4)); (rp / rp.min(axis=1).values[:, None]).T.plot(kind="bar", ax=plt.gca())
plt.ylabel("NME relative to best group"); plt.title("Disparity per perturbation"); plt.tight_layout()
plt.savefig(f"{OUT}/fig_disparity_by_perturbation.png", dpi=200); plt.show()

## 8. Download results
Send `results.zip` back to continue with the paper.

In [ ]:
import shutil; shutil.make_archive("results", "zip", OUT)
try:
    from google.colab import files; files.download("results.zip")
except Exception: print("Download results.zip from the file panel on the left.")